# Gold | Promoção `gold_staging` → `gold`
Roda **somente depois** de `01_validar_qualidade_gold` passar. Cada tabela é substituída por `CREATE OR REPLACE ... AS SELECT`
(atômico por tabela no Delta). A atomicidade entre tabelas não é garantida: se uma promoção falhar no meio, o erro é registrado
e o job falha — reexecute o job inteiro.

In [0]:
%run ../../Monitoring/_common_monitoring

In [0]:
_m = iniciar_etapa()
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")
tabelas = [r["tableName"] for r in spark.sql("SHOW TABLES IN workspace.gold_staging").collect()]
try:
    for t in tabelas:
        spark.sql(f"CREATE OR REPLACE TABLE workspace.gold.{t} AS SELECT * FROM workspace.gold_staging.{t}")
        print("promovida:", t)
except Exception as exc:
    registrar_execucao("GOLD", "promocao", _m, status="FALHA", erro=exc)
    raise
registrar_execucao("GOLD", "promocao", _m, row_count=len(tabelas))